# ERCOT Data Preprocessing & Feature Engineering

Based on inspecting the actual data, here is what we found and need to fix:

| Issue | Details | Fix |
|-------|---------|-----|
| 9 missing weather rows | 8 are DST spring-forward hours (2 AM disappears in March) + 1 stray 2026-01-01 row | Interpolate DST, drop stray row |
| `datetime` is a string | pandas read it as object dtype | Parse to proper datetime |
| Calendar cols are float | `hour`, `month` etc. saved as 1.0, 2.0 | Cast to int |
| 5 extreme load rows | Aug 2023 & 2024 heat waves ~85,000 MW | Keep — these are real events |

Then we engineer new features on top of the clean data.

## Step 1: Imports & Load Data

In [32]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/bestek/PycharmProjects/ercot-electricity-demand-forecasting/processed_data/ercot_full_dataset_2018_2025.csv")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

Loaded: 70,136 rows x 48 columns


,datetime,COAST_load_mw,EAST_load_mw,FWEST_load_mw,NORTH_load_mw,NCENT_load_mw,SOUTH_load_mw,SCENT_load_mw,WEST_load_mw,ERCOT_total_load_mw,...,NORTH_solar_radiation_wm2,SCENT_solar_radiation_wm2,SOUTH_solar_radiation_wm2,WEST_solar_radiation_wm2,hour,day_of_week,month,year,is_weekend,is_holiday
0,2018-01-01 01:00:00,11425.979115,1852.663959,2823.409245,1135.360907,18584.343617,3831.649454,9151.190703,1762.472684,50567.069682,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,2018.0,0.0,1.0
1,2018-01-01 02:00:00,11408.418023,1850.169452,2809.745403,1136.630855,18524.141392,3988.271046,9144.993712,1754.718094,50617.087977,...,0.0,0.0,0.0,0.0,2.0,0.0,1.0,2018.0,0.0,1.0
2,2018-01-01 03:00:00,11405.198365,1858.269586,2797.802576,1135.930264,18532.056616,4076.086451,9141.036615,1747.919615,50694.300087,...,0.0,0.0,0.0,0.0,3.0,0.0,1.0,2018.0,0.0,1.0


## Step 2: Fix datetime Column

Right now `datetime` is just a string like `'2018-01-01 01:00:00'`. 
We need it as a real datetime so we can do time-based operations on it.

In [33]:
df['datetime'] = pd.to_datetime(df['datetime'])

print("datetime dtype is now:", df['datetime'].dtype)
print("Sample:", df['datetime'].iloc[0])

datetime dtype is now: datetime64[ns]
Sample: 2018-01-01 01:00:00


## Step 3: Drop the Stray 2026-01-01 Row

Our data is supposed to cover 2018–2025. The `24:00` to `00:00` conversion 
turned `2025-12-31 24:00` into `2026-01-01 00:00`, which has no weather data.
We drop it since it's outside our study period.

In [34]:
before = len(df)
df = df[df['datetime'].dt.year <= 2025].copy()
print(f"Dropped {before - len(df)} row(s) outside 2018–2025")
print(f"Rows remaining: {len(df):,}")

Dropped 1 row(s) outside 2018–2025
Rows remaining: 70,135


## Step 4: Handle DST Duplicates

In [ ]:
# Check duplicates before fix
dupes_before = df.duplicated(subset='datetime', keep=False).sum()
print(f"Duplicate rows before fix: {dupes_before}")

# Average all numeric columns for duplicate timestamps
numeric_cols = df.select_dtypes(include='number').columns.tolist()
df = (
    df.groupby('datetime', as_index=False)[numeric_cols]
    .mean()
)

# Re-sort by time
df = df.sort_values('datetime').reset_index(drop=True)

dupes_after = df.duplicated(subset='datetime', keep=False).sum()
print(f"Duplicate rows after fix:  {dupes_after}")
print(f"Total rows now: {len(df):,}")

Duplicate rows before fix: 32
Duplicate rows after fix:  0 (should be 0)
Total rows now: 70,119


## Step 4: Fix the DST Missing Hours

Every year in March, clocks spring forward from 2:00 AM to 3:00 AM — so 
**2:00 AM literally doesn't exist** that day. ERCOT still has a load value 
for it (they interpolate internally), but our weather API has no data for 
a time that doesn't exist in local time.

Fix: **linear interpolation** — fill the missing weather value as the midpoint 
between 1:00 AM and 3:00 AM. This is standard practice for DST gaps.

In [37]:
# First: show which rows are missing BEFORE any fix
dst_mask = df['COAST_temperature_f'].isnull()
print(f"Missing rows BEFORE fix: {dst_mask.sum()}")
print(df[dst_mask][['datetime', 'COAST_temperature_f', 'hour']].to_string())

# Step 4a: Rebuild calendar columns directly from datetime
# Do this FIRST so they are never NaN regardless of weather gaps
import holidays
us_holidays = holidays.US(state='TX', years=range(2018, 2026))

df['hour']        = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month']       = df['datetime'].dt.month
df['year']        = df['datetime'].dt.year
df['is_weekend']  = (df['datetime'].dt.dayofweek >= 5).astype(int)
df['is_holiday']  = df['datetime'].dt.date.astype(str).isin(
    [str(d) for d in us_holidays.keys()]
).astype(int)

# Step 4b: Interpolate weather columns only
weather_cols = [c for c in df.columns if c not in 
                ['datetime','COAST_load_mw','EAST_load_mw','FWEST_load_mw',
                 'NORTH_load_mw','NCENT_load_mw','SOUTH_load_mw','SCENT_load_mw',
                 'WEST_load_mw','ERCOT_total_load_mw',
                 'hour','day_of_week','month','year','is_weekend','is_holiday']]

df[weather_cols] = df[weather_cols].interpolate(method='linear')

# Verify
print(f"\nMissing rows AFTER fix: {df.isnull().sum().sum()}")
print("\nDST rows now filled (March 2:00 AM each year):")
dst_filled = df[(df['month'] == 3) & (df['hour'] == 2) & 
                df['datetime'].isin(pd.to_datetime([
                    '2018-03-11 02:00','2019-03-10 02:00','2020-03-08 02:00',
                    '2021-03-14 02:00','2022-03-13 02:00','2023-03-12 02:00',
                    '2024-03-10 02:00','2025-03-09 02:00'
                ]))]
print(dst_filled[['datetime','COAST_temperature_f','hour']].to_string())

Missing rows BEFORE fix: 8
                 datetime  COAST_temperature_f  hour
1657  2018-03-11 02:00:00                  NaN   NaN
10392 2019-03-10 02:00:00                  NaN   NaN
19127 2020-03-08 02:00:00                  NaN   NaN
28030 2021-03-14 02:00:00                  NaN   NaN
36765 2022-03-13 02:00:00                  NaN   NaN
45500 2023-03-12 02:00:00                  NaN   NaN
54235 2024-03-10 02:00:00                  NaN   NaN
62970 2025-03-09 02:00:00                  NaN   NaN

Missing rows AFTER fix: 0

DST rows now filled (March 2:00 AM each year):
                 datetime  COAST_temperature_f  hour
1657  2018-03-11 02:00:00               69.800     2
10392 2019-03-10 02:00:00               64.310     2
19127 2020-03-08 02:00:00               51.710     2
28030 2021-03-14 02:00:00               68.360     2
36765 2022-03-13 02:00:00               38.795     2
45500 2023-03-12 02:00:00               70.565     2
54235 2024-03-10 02:00:00               48.335    

## Step 5: Verify Clean Data

In [38]:
print("=== MISSING VALUES ===")
print(df.isnull().sum().sum(), 'total missing values')
print()
print("=== CALENDAR COLUMN DTYPES ===")
int_cols = ['hour','day_of_week','month','year','is_weekend','is_holiday']
print(df[int_cols].dtypes)
print()
print("=== SAMPLE ===")
df[['datetime','ERCOT_total_load_mw','NCENT_temperature_f','hour','month','is_holiday']].head()

=== MISSING VALUES ===
0 total missing values

=== CALENDAR COLUMN DTYPES ===
hour           int32
day_of_week    int32
month          int32
year           int32
is_weekend     int64
is_holiday     int64
dtype: object

=== SAMPLE ===


,datetime,ERCOT_total_load_mw,NCENT_temperature_f,hour,month,is_holiday
0,2018-01-01 01:00:00,50567.069682,20.480000,1,1,1
1,2018-01-01 02:00:00,50617.087977,19.850000,2,1,1
2,2018-01-01 03:00:00,50694.300087,19.220001,3,1,1
3,2018-01-01 04:00:00,50999.591693,18.500000,4,1,1
4,2018-01-01 05:00:00,51723.732017,18.230000,5,1,1


---
# Feature Engineering

Now that the data is clean, we engineer new features. 
These are derived columns that help the models learn patterns 
that aren't obvious from raw values alone.

We'll add features in 4 groups:
1. **Cyclic time features** — encode hour/month as sine/cosine waves
2. **Lag features** — yesterday's load, last week's load
3. **Rolling averages** — smoothed recent load trends
4. **Heat index** — combines temperature + humidity into one comfort metric

## Step 6: Cyclic Time Encoding

**Why this matters:** Hour 23 and hour 0 are only 1 hour apart, but numerically 
they're 23 apart. A model treating hour as a plain number gets confused — it 
thinks midnight and 11 PM are very different when they're actually neighbors.

**The fix:** encode hour (and month) as a point on a circle using sine and cosine. 
This way hour 23 and hour 0 end up close together in the model's feature space.

```
hour_sin = sin(2π × hour / 24)
hour_cos = cos(2π × hour / 24)
```

In [39]:
# Hour of day (cycles every 24 hours)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Month of year (cycles every 12 months)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Day of week (cycles every 7 days)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

print("Cyclic features added: hour_sin, hour_cos, month_sin, month_cos, dow_sin, dow_cos")
print(df[['hour','hour_sin','hour_cos']].iloc[:5].round(3))

Cyclic features added: hour_sin, hour_cos, month_sin, month_cos, dow_sin, dow_cos
   hour  hour_sin  hour_cos
0     1     0.259     0.966
1     2     0.500     0.866
2     3     0.707     0.707
3     4     0.866     0.500
4     5     0.966     0.259


## Step 7: Lag Features

**Why this matters:** Electricity demand right now is heavily influenced by 
what it was 24 hours ago (same hour yesterday) and 168 hours ago (same hour 
last week). These are called **lag features** — we're giving the model 
a direct memory of recent history.

We create lags on `ERCOT_total_load_mw` as the main target, 
and also on `NCENT_temperature_f` as the most influential weather variable.

In [40]:
# Sort by time first — critical for lag features to be correct
df = df.sort_values('datetime').reset_index(drop=True)

# Load lags
df['load_lag_24h']  = df['ERCOT_total_load_mw'].shift(24)   # same hour yesterday
df['load_lag_48h']  = df['ERCOT_total_load_mw'].shift(48)   # same hour 2 days ago
df['load_lag_168h'] = df['ERCOT_total_load_mw'].shift(168)  # same hour last week

# Temperature lag for ERCOT-wide proxy (using NCENT = Dallas, largest zone)
df['temp_lag_24h']  = df['NCENT_temperature_f'].shift(24)

print("Lag features added!")
print(f"NaNs introduced by lags (expected — first 168 rows): {df['load_lag_168h'].isnull().sum()}")
df[['datetime','ERCOT_total_load_mw','load_lag_24h','load_lag_168h']].iloc[165:172]

Lag features added!
NaNs introduced by lags (expected — first 168 rows): 168


,datetime,ERCOT_total_load_mw,load_lag_24h,load_lag_168h
165,2018-01-07 22:00:00,35880.205093,36921.360889,NaN
166,2018-01-07 23:00:00,33992.684812,35772.332555,NaN
167,2018-01-08 00:00:00,32063.404932,34314.168708,NaN
168,2018-01-08 01:00:00,30767.007132,33025.925584,50567.069682
169,2018-01-08 02:00:00,30231.165136,32160.065962,50617.087977
170,2018-01-08 03:00:00,30174.984050,31691.973813,50694.300087
171,2018-01-08 04:00:00,30740.039419,31482.431729,50999.591693


## Step 8: Rolling Average Features

**Why this matters:** A rolling average smooths out noise and captures 
the recent trend. For example, a 24-hour rolling average tells the model 
whether we're in a generally high-demand period (summer heat wave) 
or a low-demand period (mild spring).

`min_periods=1` means we start computing even before we have a full window — 
useful for the beginning of the dataset.

In [41]:
df['load_rolling_24h_mean'] = (
    df['ERCOT_total_load_mw']
    .shift(1)                          # shift by 1 so we don't include current hour
    .rolling(window=24, min_periods=1)
    .mean()
)

df['load_rolling_168h_mean'] = (
    df['ERCOT_total_load_mw']
    .shift(1)
    .rolling(window=168, min_periods=1)
    .mean()
)

print("Rolling features added!")
df[['datetime','ERCOT_total_load_mw','load_rolling_24h_mean','load_rolling_168h_mean']].iloc[20:25]

Rolling features added!


,datetime,ERCOT_total_load_mw,load_rolling_24h_mean,load_rolling_168h_mean
20,2018-01-01 21:00:00,57246.956558,53432.530057,53432.530057
21,2018-01-01 22:00:00,56339.310019,53614.169414,53614.169414
22,2018-01-01 23:00:00,54402.041230,53738.039441,53738.039441
23,2018-01-02 00:00:00,52550.803685,53766.909084,53766.909084
24,2018-01-02 01:00:00,51180.170308,53716.238026,53716.238026


## Step 9: Heat Index Feature

**Why this matters:** A 95°F day with 80% humidity feels much worse than 
a 95°F day with 30% humidity — and AC usage reflects that. The heat index 
combines temperature and humidity into one 'feels like' number that better 
predicts electricity demand than temperature alone.

We compute it for each zone using the standard Rothfusz formula 
(same one used by the National Weather Service).

In [42]:
def heat_index(T, RH):
    """
    Rothfusz heat index formula.
    T  = temperature in Fahrenheit
    RH = relative humidity in percent (0–100)
    Returns 'feels like' temperature in Fahrenheit.
    Only meaningful when T >= 80°F — below that we just return T.
    """
    HI = (
        -42.379
        + 2.04901523  * T
        + 10.14333127 * RH
        - 0.22475541  * T * RH
        - 0.00683783  * T**2
        - 0.05481717  * RH**2
        + 0.00122874  * T**2 * RH
        + 0.00085282  * T * RH**2
        - 0.00000199  * T**2 * RH**2
    )
    # Only apply when hot enough for heat index to be meaningful
    return np.where(T >= 80, HI, T)

ZONES = ['COAST','EAST','FWEST','NORTH','NCENT','SOUTH','SCENT','WEST']

for zone in ZONES:
    df[f'{zone}_heat_index_f'] = heat_index(
        df[f'{zone}_temperature_f'],
        df[f'{zone}_relative_humidity_pct']
    )

print("Heat index features added for all 8 zones!")
# Show difference between temp and heat index on a hot day
hot_day = df[df['NCENT_temperature_f'] > 100].head(3)
print(hot_day[['datetime','NCENT_temperature_f','NCENT_relative_humidity_pct','NCENT_heat_index_f']].to_string())

Heat index features added for all 8 zones!
                datetime  NCENT_temperature_f  NCENT_relative_humidity_pct  NCENT_heat_index_f
3663 2018-06-02 17:00:00            100.03999                    35.844894          106.193080
4263 2018-06-27 17:00:00            100.40000                    31.498861          103.937326
4264 2018-06-27 18:00:00            100.40000                    30.427803          103.263336


## Step 10: Final Summary

In [43]:
print(f"Final dataset shape: {df.shape}")
print(f"Total features: {df.shape[1]}")
print()

# Group columns by type for a clean overview
groups = {
    'Target (load)':      [c for c in df.columns if 'load_mw' in c],
    'Temperature':        [c for c in df.columns if 'temperature' in c],
    'Heat Index':         [c for c in df.columns if 'heat_index' in c],
    'Wind':               [c for c in df.columns if 'wind_speed' in c],
    'Solar':              [c for c in df.columns if 'solar' in c],
    'Humidity':           [c for c in df.columns if 'humidity' in c],
    'Calendar (raw)':     ['hour','day_of_week','month','year','is_weekend','is_holiday'],
    'Cyclic encoding':    [c for c in df.columns if any(x in c for x in ['_sin','_cos'])],
    'Lag features':       [c for c in df.columns if 'lag' in c],
    'Rolling averages':   [c for c in df.columns if 'rolling' in c],
}

for group, cols in groups.items():
    print(f"  {group} ({len(cols)}): {cols}")

Final dataset shape: (70119, 68)
Total features: 68

  Target (load) (9): ['COAST_load_mw', 'EAST_load_mw', 'FWEST_load_mw', 'NORTH_load_mw', 'NCENT_load_mw', 'SOUTH_load_mw', 'SCENT_load_mw', 'WEST_load_mw', 'ERCOT_total_load_mw']
  Temperature (8): ['COAST_temperature_f', 'EAST_temperature_f', 'FWEST_temperature_f', 'NCENT_temperature_f', 'NORTH_temperature_f', 'SCENT_temperature_f', 'SOUTH_temperature_f', 'WEST_temperature_f']
  Heat Index (8): ['COAST_heat_index_f', 'EAST_heat_index_f', 'FWEST_heat_index_f', 'NORTH_heat_index_f', 'NCENT_heat_index_f', 'SOUTH_heat_index_f', 'SCENT_heat_index_f', 'WEST_heat_index_f']
  Wind (8): ['COAST_wind_speed_mph', 'EAST_wind_speed_mph', 'FWEST_wind_speed_mph', 'NCENT_wind_speed_mph', 'NORTH_wind_speed_mph', 'SCENT_wind_speed_mph', 'SOUTH_wind_speed_mph', 'WEST_wind_speed_mph']
  Solar (8): ['COAST_solar_radiation_wm2', 'EAST_solar_radiation_wm2', 'FWEST_solar_radiation_wm2', 'NCENT_solar_radiation_wm2', 'NORTH_solar_radiation_wm2', 'SCENT_solar

## Step 11: Save Clean Dataset + Splits

In [44]:
# Full clean dataset
df.to_csv('ercot_processed_2018_2025.csv', index=False)
print(f"Saved: ercot_processed_2018_2025.csv  ({df.shape[0]:,} rows x {df.shape[1]} cols)")

# Train / validate / test splits
train = df[df['year'] <= 2023]
val   = df[df['year'] == 2024]
test  = df[df['year'] == 2025]

train.to_csv('ercot_train_processed.csv', index=False)
val.to_csv('ercot_validate_processed.csv', index=False)
test.to_csv('ercot_test_processed.csv', index=False)

print(f"Saved: ercot_train_processed.csv      ({len(train):,} rows)")
print(f"Saved: ercot_validate_processed.csv   ({len(val):,} rows)")
print(f"Saved: ercot_test_processed.csv       ({len(test):,} rows)")
print("\nReady for EDA and modeling!")

Saved: ercot_processed_2018_2025.csv  (70,119 rows x 68 cols)
Saved: ercot_train_processed.csv      (52,577 rows)
Saved: ercot_validate_processed.csv   (8,783 rows)
Saved: ercot_test_processed.csv       (8,759 rows)

Ready for EDA and modeling!
